In [81]:
import pandas as pd
import numpy as np
import seaborn as sns
import scipy.stats as stats
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, root_mean_squared_error, r2_score
from typing import List, Dict, Any
import re
pd.set_option('display.max_columns', None)

In [82]:
df_datos : pd.DataFrame = pd.read_pickle("../resources/raw_data_10_years")
df_datos
# df_datos.to_csv("../resources/raw_data_10_years.csv", index=False)
# df_datos.to_json("../resources/raw_data_10_years.json", orient="records", lines=True)

,fecha,indicativo,nombre,provincia,altitud,tmed,prec,tmin,horatmin,tmax,horatmax,hrMedia,hrMax,horaHrMax,hrMin,horaHrMin,pintMax,dir,velmedia,racha,horaracha,presMax,horaPresMax,presMin,horaPresMin,sol,horaPIntMax
0,2016-08-08,7250C,ABANILLA,MURCIA,174,"25,0","0,0","17,3",04:22,"32,7",Varias,46,80,00:00,32,13:30,"0,0",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2016-08-08,0255B,SANTA SUSANNA,BARCELONA,40,"22,9","0,0","16,2",02:50,"29,6",12:00,48,69,23:40,34,09:10,"0,0",22,"1,9","8,6",13:10,NaN,NaN,NaN,NaN,NaN,NaN
2,2016-08-08,5612B,LA RODA DE ANDALUCÍA,SEVILLA,410,"26,4","0,0","19,5",05:40,"33,2",14:00,38,69,05:40,24,Varias,"0,0",18,"5,3","14,2",16:20,"973,5",08,"970,3",18,NaN,NaN
3,2016-08-08,2885K,FRESNO DE SAYAGO,ZAMORA,804,"25,8","0,0","15,7",05:13,"36,0",15:53,26,60,Varias,13,13:20,"0,0",05,"4,4","11,7",08:40,NaN,NaN,NaN,NaN,NaN,NaN
4,2016-08-08,8492X,ATZENETA DEL MAESTRAT,CASTELLON,420,"24,5","0,0","14,7",05:09,"34,3",14:31,41,89,05:10,27,Varias,"0,0",19,"2,8","8,9",15:10,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3042388,2026-07-24,C665T,VALLESECO,LAS PALMAS,900,"19,0","0,0","15,1",06:37,"22,8",13:54,82,96,07:00,66,02:30,"0,0",34,"1,9","5,3",12:30,NaN,NaN,NaN,NaN,NaN,NaN
3042389,2026-07-24,4061X,QUINTANAR DE LA ORDEN,TOLEDO,691,"28,4","0,0","21,7",23:59,"35,0",14:30,24,39,07:20,14,Varias,"0,0",20,"3,1","13,1",13:20,"937,5",07,"934,6",Varias,NaN,NaN
3042390,2026-07-24,2096B,LICERAS,SORIA,1150,"22,9","0,0","16,1",23:02,"29,7",13:53,35,50,06:40,22,Varias,"0,0",30,"5,8","10,8",15:00,NaN,NaN,NaN,NaN,NaN,NaN
3042391,2026-07-24,2140A,ALDEANUEVA DE SERREZUELA,SEGOVIA,1135,"21,7","0,0","16,0",23:56,"27,4",14:40,39,65,05:50,23,Varias,"0,0",27,"5,0","13,6",15:10,NaN,NaN,NaN,NaN,NaN,NaN


In [83]:
display(df_datos.info(show_counts=True))
df_datos.isnull().sum()

<class 'pandas.DataFrame'>
RangeIndex: 3042393 entries, 0 to 3042392
Data columns (total 27 columns):
 #   Column       Non-Null Count    Dtype
---  ------       --------------    -----
 0   fecha        3042393 non-null  str  
 1   indicativo   3042393 non-null  str  
 2   nombre       3042393 non-null  str  
 3   provincia    3042393 non-null  str  
 4   altitud      3042393 non-null  str  
 5   tmed         2981562 non-null  str  
 6   prec         2955513 non-null  str  
 7   tmin         2982448 non-null  str  
 8   horatmin     2962430 non-null  str  
 9   tmax         2983336 non-null  str  
 10  horatmax     2964127 non-null  str  
 11  hrMedia      2939927 non-null  str  
 12  hrMax        2900286 non-null  str  
 13  horaHrMax    2899041 non-null  str  
 14  hrMin        2900586 non-null  str  
 15  horaHrMin    2899368 non-null  str  
 16  pintMax      2922754 non-null  str  
 17  dir          2497819 non-null  str  
 18  velmedia     2508153 non-null  str  
 19  racha      

None

fecha                0
indicativo           0
nombre               0
provincia            0
altitud              0
tmed             60831
prec             86880
tmin             59945
horatmin         79963
tmax             59057
horatmax         78266
hrMedia         102466
hrMax           142107
horaHrMax       143352
hrMin           141807
horaHrMin       143025
pintMax         119639
dir             544574
velmedia        534240
racha           544518
horaracha       544677
presMax        2269393
horaPresMax    2269420
presMin        2269409
horaPresMin    2269459
sol            2510690
horaPIntMax    2230420
dtype: int64

In [84]:
df_especiales = pd.DataFrame({
    "acum": [(df_datos[col].astype(str).str.strip().str.lower() == "acum").sum() for col in df_datos.columns],
    "varias": [(df_datos[col].astype(str).str.strip().str.lower() == "varias").sum() for col in df_datos.columns],
    "ip": [(df_datos[col].astype(str).str.strip().str.lower() == "ip").sum() for col in df_datos.columns],
}, index=df_datos.columns)

# Filtra solo las columnas que realmente tienen alguno de los dos
display(df_especiales[(df_especiales["acum"] > 0) | (df_especiales["varias"] > 0) | (df_especiales["ip"] > 0)])

,acum,varias,ip
prec,387,0,18506
horatmin,0,162424,0
horatmax,0,56990,0
horaHrMax,0,978845,0
horaHrMin,0,282259,0
horaracha,0,150013,0
horaPresMax,0,163295,0
horaPresMin,0,94645,0
horaPIntMax,0,232105,0


In [85]:
import pandas as pd
import numpy as np
import re

def normalizar_hora(valor):
    # Comprobamos si es nulo antes de convertir a string
    if pd.isna(valor):
        return np.nan
    valor = str(valor).strip()
    if valor in ("<NA>", "nan", "NaN", "None", ""):
        return np.nan
    if re.fullmatch(r"\d{1,2}", valor):
        return f"{int(valor):02d}:00"
    return valor

def convertir_datos_aemet(df):
    df = df.copy()

    # Fecha a datetime
    df["fecha"] = pd.to_datetime(df["fecha"], format="%Y-%m-%d", errors="coerce")

    # Categóricas / texto
    for col in ["indicativo", "provincia", "nombre"]:
        if col in df.columns:
            df[col] = df[col].astype("category")

    columnas_float = [
        "tmed", "prec", "tmin", "tmax", "hrMedia", "pintMax",
        "velmedia", "racha", "presMax", "presMin", "sol"
    ]

    columnas_hora = [
        "horatmin", "horatmax", "horaHrMax", "horaHrMin",
        "horaracha", "horaPresMax", "horaPresMin", "horaPIntMax"
    ]

    columnas_a_revisar = [c for c in columnas_float + columnas_hora if c in df.columns]

    # Detectar Acum / Varias / Ip
    mask_acum = pd.Series(False, index=df.index)
    mask_varias = pd.Series(False, index=df.index)
    mask_ip = pd.Series(False, index=df.index)

    for col in columnas_a_revisar:
        serie = df[col].apply(lambda x: str(x).strip().lower() if pd.notna(x) else np.nan)
        mask_acum |= (serie == "acum")
        mask_varias |= (serie == "varias")
        mask_ip |= (serie == "ip")

    df["precAcum"] = mask_acum.fillna(False)
    df["variasHoras"] = mask_varias.fillna(False)
    df["precIp"] = mask_ip.fillna(False)

    # Limpieza numérica
    def limpiar_numero(valor):
        if pd.isna(valor):
            return np.nan
        valor = str(valor).strip()
        if valor.lower() in ("ip",):
            return "0.05"
        if valor.lower() in ("acum", "varias", "<na>", "nan", "none", ""):
            return np.nan
        return valor.replace(",", ".")

    for col in columnas_float:
        if col in df.columns:
            df[col] = df[col].apply(limpiar_numero)
            df[col] = pd.to_numeric(df[col], errors="coerce")

    # 4. Numéricos -> float
    for col in ["altitud", "hrMax", "hrMin", "dir"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce").astype(float)

    # 5. Columnas de hora -> normalizadas a HH:MM
    for col in columnas_hora:
        if col in df.columns:
            df[col] = df[col].apply(normalizar_hora)

    return df

df_datos_limpio = convertir_datos_aemet(df_datos)

pd.set_option('display.max_columns', None)
df_datos_limpio.sample(50)

,fecha,indicativo,nombre,provincia,altitud,tmed,prec,tmin,horatmin,tmax,horatmax,hrMedia,hrMax,horaHrMax,hrMin,horaHrMin,pintMax,dir,velmedia,racha,horaracha,presMax,horaPresMax,presMin,horaPresMin,sol,horaPIntMax,precAcum,variasHoras,precIp
1733051,2022-05-02,5624X,AGUILAR DE LA FRONTERA,CORDOBA,309.0,16.3,5.8,11.2,05:39,21.4,16:26,67.0,96.0,23:20,43.0,15:20,6.0,23.0,5.0,12.5,17:30,NaN,NaN,NaN,NaN,NaN,23:45,False,False,False
271113,2017-07-05,4007Y,OSSA DE MONTIEL,ALBACETE,905.0,26.7,1.4,17.4,05:16,36.0,14:44,37.0,68.0,22:20,19.0,Varias,6.0,22.0,1.9,13.1,19:30,NaN,NaN,NaN,NaN,NaN,17:25,False,True,False
2714434,2025-06-28,1167J,"CORISCAO, PARQUE NACIONAL PICOS DE EUROPA",CANTABRIA,1722.0,19.4,NaN,15.3,00:10,23.5,14:40,56.0,90.0,00:40,48.0,15:50,NaN,3.0,0.8,3.6,12:50,840.5,12:00,839.3,Varias,NaN,NaN,False,True,False
809677,2019-04-20,1249I,OVIEDO,ASTURIAS,336.0,15.4,0.0,9.4,06:00,21.5,14:00,63.0,97.0,07:00,44.0,17:00,0.0,3.0,2.8,9.4,16:30,984.2,Varias,980.1,24:00,12.2,NaN,False,True,False
1226751,2020-09-06,8416,VALÈNCIA,VALENCIA,13.0,24.8,0.0,20.3,NaN,29.4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,12.0,NaN,False,False,False
321003,2017-09-04,2106B,CORUÑA DEL CONDE,BURGOS,955.0,19.6,0.0,14.8,05:58,24.5,16:59,57.0,80.0,07:40,42.0,17:00,0.0,99.0,1.4,5.3,Varias,NaN,NaN,NaN,NaN,NaN,NaN,False,True,False
153150,2017-02-12,2453E,GOTARRENDURA,AVILA,920.0,6.8,8.2,4.4,01:44,9.2,13:19,85.0,90.0,Varias,75.0,15:20,2.4,16.0,5.6,19.7,23:59,NaN,NaN,NaN,NaN,NaN,Varias,False,True,False
1754820,2022-05-28,0200E,"BARCELONA, FABRA",BARCELONA,408.0,24.7,0.0,18.7,23:55,30.7,12:05,51.0,NaN,NaN,NaN,NaN,0.0,32.0,4.2,12.5,04:19,969.1,00:00,961.6,24:00,13.3,NaN,False,False,False
825752,2019-05-10,0324A,RIPOLL,GIRONA,675.0,16.0,NaN,8.8,05:30,23.2,14:10,71.0,90.0,Varias,51.0,12:00,NaN,3.0,0.8,7.8,17:00,NaN,NaN,NaN,NaN,NaN,NaN,False,True,False
1115139,2020-04-24,2182C,PEDRAZA,SEGOVIA,1107.0,14.0,0.0,9.2,05:28,18.9,12:23,57.0,80.0,03:50,46.0,17:10,0.0,16.0,2.8,8.3,12:20,NaN,NaN,NaN,NaN,NaN,NaN,False,False,False


In [86]:
df_datos_pres = df_datos[["fecha","indicativo", "provincia", "presMax", "presMin"]].sort_values(by=["fecha"], ascending=[False])
df_datos_pres


,fecha,indicativo,provincia,presMax,presMin
3042392,2026-07-24,3469A,CACERES,"972,3","969,5"
3041840,2026-07-24,8072Y,VALENCIA,NaN,NaN
3041850,2026-07-24,9894Y,HUESCA,NaN,NaN
3041849,2026-07-24,4244X,BADAJOZ,"965,8","963,0"
3041848,2026-07-24,1738U,OURENSE,NaN,NaN
...,...,...,...,...,...
542,2016-08-08,3130C,GUADALAJARA,"912,9","908,8"
541,2016-08-08,0385X,GIRONA,NaN,NaN
540,2016-08-08,6127X,MALAGA,NaN,NaN
539,2016-08-08,6045X,MALAGA,NaN,NaN


In [87]:
# Resumen de presiones por fecha y provincia (conocemos si hay provincias en las que algun día no se ha registrado presión máxima o mínima en ninguna estación)
resumen_presiones = df_datos_pres.groupby(["fecha", "provincia"]).agg(
    n_estaciones_presMax=("presMax", "count"),
    n_estaciones_presMin=("presMin", "count")
).reset_index()

display(resumen_presiones)

,fecha,provincia,n_estaciones_presMax,n_estaciones_presMin
0,2016-08-08,A CORUÑA,7,7
1,2016-08-08,ALBACETE,4,4
2,2016-08-08,ALICANTE,3,3
3,2016-08-08,ALMERIA,6,6
4,2016-08-08,ARABA/ALAVA,2,2
...,...,...,...,...
196369,2026-07-24,TOLEDO,3,3
196370,2026-07-24,VALENCIA,7,7
196371,2026-07-24,VALLADOLID,2,2
196372,2026-07-24,ZAMORA,4,4


In [88]:
# Combinaciones fecha-provincia sin ninguna estación con presión
sin_presion = resumen_presiones[
    (resumen_presiones["n_estaciones_presMax"] == 0) |
    (resumen_presiones["n_estaciones_presMin"] == 0)
]
display(sin_presion)

,fecha,provincia,n_estaciones_presMax,n_estaciones_presMin
4087,2016-10-22,NAVARRA,0,0
6604,2016-12-08,CEUTA,0,0
6658,2016-12-09,CEUTA,0,0
6712,2016-12-10,CEUTA,0,0
6766,2016-12-11,CEUTA,0,0
...,...,...,...,...
191328,2026-04-21,AVILA,0,0
194137,2026-06-12,SEGOVIA,0,0
195197,2026-07-02,SEGOVIA,0,0
195250,2026-07-03,SEGOVIA,0,0


In [89]:
# Días sin presión por provincia
sin_presion.groupby("provincia").agg(
    n_dias_sin_presion=("fecha", "count")
)

,n_dias_sin_presion
provincia,
AVILA,13
CEUTA,28
LA RIOJA,10
LUGO,28
MELILLA,24
NAVARRA,1
OURENSE,25
PALENCIA,11
SEGOVIA,5


In [ ]:
df_datos_limpio.info(show_counts=True)
# df_datos_limpio.to_json("../resources/cleaned_data_10_years.json", orient="records", lines=True)
# df_datos_limpio.to_csv("../resources/cleaned_data_10_years.csv", index=False)

<class 'pandas.DataFrame'>
RangeIndex: 3042393 entries, 0 to 3042392
Data columns (total 30 columns):
 #   Column       Non-Null Count    Dtype         
---  ------       --------------    -----         
 0   fecha        3042393 non-null  datetime64[us]
 1   indicativo   3042393 non-null  category      
 2   nombre       3042393 non-null  category      
 3   provincia    3042393 non-null  category      
 4   altitud      3042393 non-null  float64       
 5   tmed         2981562 non-null  float64       
 6   prec         2955126 non-null  float64       
 7   tmin         2982448 non-null  float64       
 8   horatmin     2962430 non-null  str           
 9   tmax         2983336 non-null  float64       
 10  horatmax     2964127 non-null  str           
 11  hrMedia      2939927 non-null  float64       
 12  hrMax        2900286 non-null  float64       
 13  horaHrMax    2899041 non-null  str           
 14  hrMin        2900586 non-null  float64       
 15  horaHrMin    2899368 non-n

C:\Users\awm\AppData\Local\Temp\ipykernel_15512\4168975533.py:2: Pandas4Warning: The default 'epoch' date format is deprecated and will be removed in a future version, please use 'iso' date format instead.
  df_datos_limpio.to_json("../resources/cleaned_data_10_years.json", orient="records", lines=True)


In [91]:
df_datos_limpio.sample(50)


,fecha,indicativo,nombre,provincia,altitud,tmed,prec,tmin,horatmin,tmax,horatmax,hrMedia,hrMax,horaHrMax,hrMin,horaHrMin,pintMax,dir,velmedia,racha,horaracha,presMax,horaPresMax,presMin,horaPresMin,sol,horaPIntMax,precAcum,variasHoras,precIp
1826397,2022-08-21,7103Y,TOBARRA,ALBACETE,615.0,28.2,0.0,17.7,05:45,38.7,14:49,33.0,81.0,Varias,14.0,17:00,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,True,False
1297924,2020-12-01,0413A,MAÇANET DE CABRENYS,GIRONA,355.0,7.8,0.8,4.2,03:21,11.4,10:21,84.0,100.0,Varias,35.0,23:40,1.2,31.0,0.8,16.4,23:40,NaN,NaN,NaN,NaN,NaN,Varias,False,True,False
557388,2018-06-19,2885K,FRESNO DE SAYAGO,ZAMORA,804.0,20.8,0.0,11.5,04:57,30.2,17:29,47.0,97.0,05:00,25.0,17:30,0.0,6.0,3.9,10.6,07:00,NaN,NaN,NaN,NaN,NaN,NaN,False,False,False
4561,2016-08-13,9252X,OLITE/ERRIBERRI,NAVARRA,390.0,23.4,0.0,13.2,05:31,33.7,15:56,32.0,71.0,05:30,12.0,17:30,0.0,16.0,2.2,7.8,11:10,NaN,NaN,NaN,NaN,NaN,NaN,False,False,False
363756,2017-10-26,9302Y,TUDELA,NAVARRA,297.0,16.6,0.0,8.2,06:17,25.1,15:11,60.0,87.0,06:40,37.0,15:30,0.0,28.0,0.8,4.2,22:50,NaN,NaN,NaN,NaN,NaN,NaN,False,False,False
2354296,2024-05-03,1706A,ALLARIZ,OURENSE,492.0,11.0,3.4,7.0,03:27,15.1,13:01,79.0,87.0,21:10,69.0,14:30,1.2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Varias,False,True,False
2007104,2023-03-23,6083X,MARBELLA,MALAGA,2.0,16.6,0.0,13.1,03:06,20.1,16:06,63.0,82.0,02:10,47.0,22:00,0.0,27.0,2.8,5.8,18:40,NaN,NaN,NaN,NaN,NaN,NaN,False,False,False
1735177,2022-05-05,3547X,VALVERDE DEL FRESNO,CACERES,450.0,18.2,0.0,10.4,02:27,26.1,14:54,44.0,77.0,02:30,30.0,17:20,0.0,34.0,1.9,8.3,23:30,NaN,NaN,NaN,NaN,NaN,NaN,False,False,False
1730135,2022-04-29,2539,VALLADOLID AEROPUERTO,VALLADOLID,846.0,12.8,0.0,4.3,05:10,21.2,16:10,63.0,96.0,Varias,35.0,15:20,0.0,7.0,1.9,6.1,00:10,929.2,09:00,926.1,19:00,10.8,NaN,False,True,False
959964,2019-10-19,1037Y,ZUMARRAGA,GIPUZKOA,431.0,15.2,12.0,11.3,23:59,19.1,13:40,54.0,99.0,Varias,46.0,13:50,3.6,21.0,0.6,10.3,04:50,960.4,00:00,952.1,24:00,NaN,21:45,False,True,False
